In [3]:
import pandas as pd
import numpy as np
import os
import sys
#from sklearn.metrics import mean_absolute_error
from sklearn.metrics import max_error


import json

In [6]:
DATASET_MAX_N_OBS = {
    'autompg': 400,
    'breastcancer': 200,
    'fertility': 100,  # Small dataset
    'forest': 500,
    'housing': 500,
    'pendulum': 500,
    'qsar_aquatic_toxicity': 500,
    'servo': 100,      # Small dataset
    'stock': 500,
    'yacht_hydrodynamics': 300 , # Medium dataset
    'ENB2012_data_energy_heating': 768,
    'ENB2012_data_energy_cooling': 768,
    'real_estate': 414,
    'winequality-red': 1599,
    'winequality-white': 4898,
    'airfoil_self_noise': 1503,
    'qsar_fish_toxicity': 908,
    'Combined_Cycle_Power_Plant': 9568,
}

def get_valid_n_obs_list(data_name, requested_n_obs_list):
    """
    Filter n_obs_list based on dataset's maximum available samples
    
    Args:
        data_name: Name of the dataset
        requested_n_obs_list: List of requested sample sizes
        
    Returns:
        Filtered list of valid sample sizes
    """
    max_n_obs = DATASET_MAX_N_OBS.get(data_name, 500)
    valid_list = [n for n in requested_n_obs_list if n <= max_n_obs]
    
    return valid_list

# List of datasets (same as in experiment)
data_names = [
    'fertility', 
    'forest',
    'qsar_aquatic_toxicity', 
    'stock', 
    'yacht_hydrodynamics',
    'real_estate',
    'winequality-red',
    'winequality-white',
    'qsar_fish_toxicity',
    'Combined_Cycle_Power_Plant'
]


In [5]:
def calculate_mse_metrics(data_name, n_obs_list_full = [50,100,200,300,400,500], 
                            results_dir='../results', metrics_dir='./metrics'):
    """
    Calculate MSE for all models across different sample sizes and replications
    
    Args:
        data_name: Dataset name (e.g., 'autompg')
        n_obs_list: List of sample sizes to analyze
        save: Whether to save results to CSV
        results_dir: Directory containing experiment results
        metrics_dir: Directory to save metrics (if None, won't save even if save=True)
    
    Returns:
        DataFrame with columns: data, n_obs, r, base_rf, rf10, rf100, srf_normal_*, srf_hypsec_*
    """
    n_obs_list = get_valid_n_obs_list(data_name, n_obs_list_full)
    # List to store results
    results_list = []
    
    # Define all model names (use original names for reading)
    baseline_model_names = ['rf10', 'rf20', 'rf50', 'rf_100','gp']
    srf_model_names = ['srf_normal_EST_PD','srf_hypsec_EST_PD']
    
    # Iterate through all sample sizes and replications
    for n_obs in n_obs_list:
        for r in range(100):  # 100 replications
            baseline_pred_file = f'{results_dir}/{data_name}/predictions/{data_name}_n{n_obs}_r{r}.csv'
            srf_pred_file = f'{results_dir}/{data_name}/EST_PD_predictions_noCV/{data_name}_n{n_obs}_r{r}.csv'
            
            # Check if file exists
            if not os.path.exists(baseline_pred_file):
                continue
            
            try:
                # Load predictions
                baseline_pred_df = pd.read_csv(baseline_pred_file)
                srf_pred_df = pd.read_csv(srf_pred_file)
                y_test = baseline_pred_df['y_test'].values
                
                # Create result dictionary for this replication
                result = {
                    'data': data_name,
                    'n_obs': n_obs,
                    'r': r
                }
                
                # Calculate MSE for each model
                for model in baseline_model_names:
                    pred_col = f'{model}_pred'
                    
                    if pred_col in baseline_pred_df.columns:
                        y_pred = baseline_pred_df[pred_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(y_pred)):
                            result[model] = np.nan
                        else:
                            result[model] = max_error(y_pred, y_test)
                    else:
                        result[model] = np.nan
                for srf_model in srf_model_names:
                    pred_col = f'{srf_model}_pred'
                    
                    if pred_col in srf_pred_df.columns:
                        y_pred = srf_pred_df[pred_col].values
                        
                        # Check for NaN values
                        if np.any(np.isnan(y_pred)):
                            result[srf_model] = np.nan
                        else:
                            result[srf_model] = max_error(y_pred, y_test)
                    else:
                        result[srf_model] = np.nan
                        
                
                results_list.append(result)
                
            except Exception as e:
                print(f"Error processing {data_name} n={n_obs} r={r}: {e}")
                continue
    
    # Create DataFrame
    results_df = pd.DataFrame(results_list)
    
    # Rename rf_full to rf100
    model_names_display = [m if m != 'rf_100' else 'rf100' for m in baseline_model_names]
    # combine srf_normal_EST_PD and srf_hypsec_EST_PD into baseline_model_names
    results_df.rename(columns={'rf_100': 'rf100'}, inplace=True)
    model_names_display.append('srf_normal_EST_PD')
    model_names_display.append('srf_hypsec_EST_PD')
    # Reorder columns
    cols_order = ['data', 'n_obs', 'r'] + model_names_display
    results_df = results_df[cols_order]
    results_df.rename(columns={
        'data':'Data',
        'rf10':'RF(10)',
        'rf20':'RF(20)',
        'rf50':'RF(50)',
        'rf100': 'RF(100)',
        'gp':'GP',
        'srf_normal_EST_PD':'EST-PD(Norm)',
        'srf_hypsec_EST_PD':'EST-PD(Hypsec)'}, inplace=True)
    
    # Save if requested and metrics_dir is not None
    if metrics_dir is not None:
        save_path = f'{metrics_dir}/{data_name}/{data_name}_ME.csv'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        results_df.to_csv(save_path, index=False)
        print(f"Saved metrics to: {save_path}")
    
    else:
        return results_df

In [6]:
for data in data_names:
    calculate_mse_metrics(data_name=data,metrics_dir= './metric_ME')

Saved metrics to: ./metric_ME/fertility/fertility_ME.csv
Saved metrics to: ./metric_ME/forest/forest_ME.csv
Saved metrics to: ./metric_ME/qsar_aquatic_toxicity/qsar_aquatic_toxicity_ME.csv
Saved metrics to: ./metric_ME/stock/stock_ME.csv
Saved metrics to: ./metric_ME/yacht_hydrodynamics/yacht_hydrodynamics_ME.csv
Saved metrics to: ./metric_ME/real_estate/real_estate_ME.csv
Saved metrics to: ./metric_ME/winequality-red/winequality-red_ME.csv
Saved metrics to: ./metric_ME/winequality-white/winequality-white_ME.csv
Saved metrics to: ./metric_ME/qsar_fish_toxicity/qsar_fish_toxicity_ME.csv
Saved metrics to: ./metric_ME/Combined_Cycle_Power_Plant/Combined_Cycle_Power_Plant_ME.csv


In [4]:
def combine_all_data_metrics(metric,data_list,metrics_path,save_path=None):
    metric_df = pd.DataFrame()
    for data_name in data_list:
        metric_df = pd.concat([metric_df,pd.read_csv(f'{metrics_path}/{data_name}/{data_name}_{metric}.csv')])

    if save_path is not None:
        metric_df.to_csv(f'{save_path}/{metric}_combined_all_data.csv',index=False)
    else:
        return metric_df

In [7]:
combine_all_data_metrics(metric='ME',data_list=data_names,
                                        metrics_path= './metric_ME',
                                        save_path= './metric_ME')

In [8]:
combined_ME = pd.read_csv('./metric_ME/ME_combined_all_data.csv')

In [9]:
combined_ME

,Data,n_obs,r,RF(10),RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,fertility,50,0,0.555,0.567500,0.598200,0.576100,0.590400,0.531941,0.528430
1,fertility,50,1,0.566,0.592370,0.594122,0.608800,0.651000,0.557441,0.557349
2,fertility,50,2,0.495,0.481861,0.501000,0.479280,0.424257,0.412246,0.416954
3,fertility,50,3,0.524,0.569000,0.564800,0.546395,0.621000,0.541881,0.542253
4,fertility,50,4,0.494,0.492417,0.479600,0.472100,0.445600,0.445185,0.444154
...,...,...,...,...,...,...,...,...,...,...
5295,Combined_Cycle_Power_Plant,500,95,41.011,41.073636,42.504806,42.757303,41.337986,43.087600,43.207350
5296,Combined_Cycle_Power_Plant,500,96,52.171,50.991556,50.529500,50.950350,47.972712,50.375200,50.386500
5297,Combined_Cycle_Power_Plant,500,97,47.134,45.913500,45.909600,45.934200,47.870505,47.656850,47.746780
5298,Combined_Cycle_Power_Plant,500,98,46.242,46.221167,45.282667,45.000000,44.296269,46.295950,46.505050


In [10]:
def calculate_percentage(df):
    pi_df = pd.DataFrame()
    pi_df['Data'] = df['Data']
    pi_df['N'] = df['N']
    pi_df['P'] = df['P']
    pi_df['n'] = df['n']
    pi_df['r'] = df['r']
    
    pi_df['RF(20)'] = round((df['RF(10)'] - df['RF(20)'])/df['RF(10)']*100,4)
    pi_df['RF(50)'] = round((df['RF(10)'] - df['RF(50)'])/df['RF(10)']*100,4)
    pi_df['RF(100)'] = round((df['RF(10)'] - df['RF(100)'])/df['RF(10)']*100,4)
    pi_df['GP'] = round((df['RF(10)'] - df['GP'])/df['RF(10)']*100,4)

    pi_df['EST-PD(Norm)'] = round((df['RF(10)'] - df['EST-PD(Norm)'])/df['RF(10)']*100,4)
    pi_df['EST-PD(Hypsec)'] = round((df['RF(10)'] - df['EST-PD(Hypsec)'])/df['RF(10)']*100,4)

    return pi_df

In [11]:
def add_data_info(data_names):
    data_info = pd.DataFrame()
    p_list = []
    n_list = []
    for data_name in data_names:
        data = pd.read_csv(f'../../data/{data_name}.csv')
        p_list.append(data.shape[1]-1)
        n_list.append(data.shape[0])
    data_info['Data'] = data_names
    data_info['P'] = p_list
    data_info['N'] = n_list
    return data_info

In [12]:
data_info = add_data_info(data_names)

In [13]:
# combine data_info with best_model_pi
combined_ME_df = pd.merge(combined_ME, data_info, on='Data', how='left')
# rename n to n_obs
combined_ME_df = combined_ME_df.rename(columns={'n_obs':'n'})
# order the columns
combined_ME_df = combined_ME_df[['Data','N','P','n','r','RF(10)','RF(20)','RF(50)','RF(100)','GP','EST-PD(Norm)','EST-PD(Hypsec)']]

data_name_mapping = {
    'Combined_Cycle_Power_Plant': 'CCPP',
    'qsar_fish_toxicity': 'Qsar Fish Toxicity',
    'real_estate': 'Real Estate',
    'yacht_hydrodynamics': 'Yacht Hydrodynamics',
    'qsar_aquatic_toxicity': 'Qsar Aquatic Toxicity',
    'fertility': 'Fertility',
    'stock': 'Stock',
    'winequality-red': 'Winequality (Red)',
    'winequality-white': 'Winequality (White)',
    'forest': 'Forest'
}
combined_ME_df['Data'] = combined_ME_df['Data'].replace(data_name_mapping)

In [14]:
pi_df = calculate_percentage(combined_ME_df)

In [19]:
PIMAD_mean = pi_df.groupby(['Data','N','P']).mean().drop(columns=['r','n']).reset_index()
PIMAD_std = pi_df.groupby(['Data','N','P']).std().drop(columns=['r','n']).reset_index()
pi_df['cases'] = 1
cased_by_n_obs = pi_df.groupby(['Data']).sum().reset_index()
PIMAD_std['cases'] = cased_by_n_obs['cases']
mean_MAD_95CI = pd.DataFrame()
mean_MAD_95CI['Data'] = PIMAD_mean['Data']
mean_MAD_95CI['N'] = PIMAD_mean['N']
mean_MAD_95CI['P'] = PIMAD_mean['P']

mean_MAD_95CI['RF(20)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(20)'], PIMAD_std['RF(20)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['RF(50)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(50)'], PIMAD_std['RF(50)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['RF(100)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['RF(100)'], PIMAD_std['RF(100)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['GP'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['GP'], PIMAD_std['GP']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['EST-PD(Norm)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['EST-PD(Norm)'], PIMAD_std['EST-PD(Norm)']/np.sqrt(PIMAD_std['cases']))]
mean_MAD_95CI['EST-PD(Hypsec)'] = [f"{m:.2f} ({s:.2f})" for m, s in zip(PIMAD_mean['EST-PD(Hypsec)'], PIMAD_std['EST-PD(Hypsec)']/np.sqrt(PIMAD_std['cases']))]

mean_best_model_pi = mean_MAD_95CI.sort_values(by='P', ascending=True)
mean_best_model_pi.drop(columns=['N'],inplace=True)

In [20]:
mean_best_model_pi

,Data,P,RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,CCPP,4,-0.12 (0.11),-0.22 (0.13),0.18 (0.13),3.03 (0.44),1.80 (0.13),1.63 (0.13)
4,Qsar Fish Toxicity,6,0.40 (0.25),0.76 (0.30),0.81 (0.31),-6.78 (0.58),1.78 (0.27),1.73 (0.27)
5,Real Estate,6,0.21 (0.29),1.25 (0.34),1.02 (0.34),-7.59 (0.67),2.44 (0.40),2.48 (0.41)
9,Yacht Hydrodynamics,6,5.18 (0.99),11.07 (1.03),12.33 (1.05),-92.55 (3.79),6.21 (0.86),5.76 (0.88)
3,Qsar Aquatic Toxicity,8,1.58 (0.26),2.04 (0.30),2.16 (0.32),1.31 (0.75),5.22 (0.28),5.33 (0.28)
1,Fertility,9,0.91 (0.47),1.39 (0.57),2.14 (0.59),-0.42 (1.16),4.61 (0.76),4.62 (0.76)
6,Stock,11,3.80 (0.33),4.18 (0.36),4.67 (0.38),-88.82 (1.71),5.89 (0.35),6.09 (0.35)
7,Winequality (Red),11,1.77 (0.22),2.36 (0.26),2.74 (0.27),1.92 (0.45),5.88 (0.27),6.03 (0.27)
8,Winequality (White),11,1.49 (0.21),2.60 (0.24),3.19 (0.23),12.51 (0.38),8.03 (0.22),8.08 (0.22)
2,Forest,12,0.74 (0.22),0.98 (0.26),1.04 (0.26),2.01 (0.40),2.65 (0.27),2.62 (0.27)


In [21]:
mean_best_model_pi.to_csv('./metric_ME/best_model_ME_pi.csv',index=False)

In [17]:
mean_ME_df=combined_ME_df.groupby(['Data','N','P']).mean().reset_index()
mean_ME_df.drop(columns=['N','r','n'],inplace=True)
mean_ME_df = mean_ME_df.sort_values(by='P', ascending=True).reset_index(drop=True)

In [ ]:
mean_ME_df=mean_ME_df.round(2)

,Data,P,RF(10),RF(20),RF(50),RF(100),GP,EST-PD(Norm),EST-PD(Hypsec)
0,CCPP,4,46.42,46.44,46.48,46.29,44.89,45.53,45.60
1,Qsar Fish Toxicity,6,5.09,5.06,5.03,5.03,5.37,4.98,4.99
2,Real Estate,6,61.06,60.90,60.54,60.65,64.75,59.71,59.66
3,Yacht Hydrodynamics,6,16.24,15.19,14.22,14.01,29.79,14.95,15.00
4,Qsar Aquatic Toxicity,8,5.49,5.39,5.37,5.36,5.26,5.19,5.18
5,Fertility,9,0.53,0.53,0.52,0.52,0.53,0.51,0.51
6,Stock,11,0.04,0.03,0.03,0.03,0.06,0.03,0.03
7,Winequality (Red),11,2.92,2.86,2.84,2.83,2.84,2.73,2.73
8,Winequality (White),11,3.78,3.71,3.67,3.65,3.28,3.46,3.46
9,Forest,12,5.72,5.67,5.65,5.65,5.58,5.55,5.55


In [22]:
mean_ME_df.to_csv('./metric_ME/best_model_ME.csv',index=False)